# 자동차 감지 및 바운딩 박스 그리기

이 노트북에서는 YOLO 모델이 감지한 자동차 주위에 바운딩 박스를 그리는 방법을 학습합니다.

In [ ]:
# 이 실습을 위해 설계된 워크벤치 이미지를 사용하지 않았다면, 아래 줄의 주석을 해제하고 실행하여 필요한 패키지들을 설치할 수 있습니다.
# !pip install --no-cache-dir --no-dependencies -r requirements.txt

from ultralytics import YOLO
from PIL import Image

In [ ]:
# 이번 실습에서는 객체 감지를 위해 YOLOv8m 모델을 사용할 것입니다.

model = YOLO("yolov8m.pt")

In [ ]:
# 테스트 이미지에 대한 모델 예측 결과를 가져옵니다.

img = "images/carImage0.jpg"
results = model.predict(img)

In [ ]:
# YOLO는 하나의 이미지뿐만 아니라 이미지 배열을 입력받을 수 있으며, 이에 대한 결과 배열을 반환합니다.  
# 현재는 이미지를 한 장만 제출했기 때문에, 결과 배열에서 첫 번째 요소만 가져오면 됩니다.

result = results[0]

In [ ]:
# 감지된 바운딩 박스의 개수를 확인합니다.

len(result.boxes)

In [ ]:
# 박스를 분석합니다.

box = result.boxes[0]
print("Object type:", box.cls)
print("Coordinates:", box.xyxy)
print("Probability:", box.conf)

In [ ]:
# 텐서(Tensor)에서 실제 값을 추출합니다.

cords = box.xyxy[0].tolist()
class_id = box.cls[0].item()
conf = box.conf[0].item()
print("Object type:", class_id)
print("Coordinates:", cords)
print("Probability:", conf)

In [ ]:
# COCO는 YOLO 모델이 이미 학습한 데이터셋입니다.  
# 이 데이터셋에서 감지 대상 객체들은 클래스(class)로 분류되어 있으며, 이 정보는 "Object type" 필드에 포함되어 있습니다.
# YOLOv8의 결과 객체에는 이러한 클래스들의 'names' 속성도 포함되어 있습니다.

print(result.names)

클래스 번호 '2'는 '자동차(car)' 객체에 해당합니다.  
따라서 결과에 나온 바운딩 박스는 감지된 자동차를 나타냅니다.  
이제 이미지 위에 해당 박스를 그려보겠습니다!

In [ ]:
# 먼저, 바운딩 박스의 좌표들을 리스트에 저장하고, 반올림 처리합니다.  
# 그런 다음, result.names 딕셔너리를 이용해 객체 클래스 ID에 해당하는 이름을 가져옵니다.

# First, put the coordinates in a list, and round them.
# Then get the name of the detected object class by ID using the result.names dictionary.

cords = box.xyxy[0].tolist()  # 바운딩 박스의 좌표를 리스트로 변환
cords = [round(x) for x in cords]  # 각 좌표 값을 반올림
class_id = result.names[box.cls[0].item()]
conf = round(box.conf[0].item(), 2)  # 감지 신뢰도(confidence) 반올림
print("Object type:", class_id)  # 객체 종류 출력
print("Coordinates:", cords)  # 바운딩 박스 좌표 출력
print("Probability:", conf)  # 감지 신뢰도 출력

In [ ]:
# Let's loop over all the boxes to extract the information.

for box in result.boxes:
  class_id = result.names[box.cls[0].item()]
  cords = box.xyxy[0].tolist()
  cords = [round(x) for x in cords]
  conf = round(box.conf[0].item(), 2)
  print("Object type:", class_id)
  print("Coordinates:", cords)
  print("Probability:", conf)
  print("---")

In [ ]:
# 이미지 위에 박스를 그리고, 감지된 객체의 클래스 이름과 확률(모델이 해당 객체를 얼마나 확신하는지를 나타냄)을 함께 표시합니다.

Image.fromarray(result.plot()[:, :, ::-1])

이제 이전 노트북에서 테스트했던 다수의 자동차가 포함된 이미지(carImage4.jpg)로 돌아가서, YOLO가 이미지 내의 모든 '자동차(car)'를 실제로 감지할 수 있는지 확인해보겠습니다.

In [ ]:
# 아래 코드는 이전 셀들에서 사용한 내용과 동일하지만, 한 번에 실행되도록 구성된 버전입니다.

results = model.predict("images/carImage4.jpg")

result = results[0]

for box in result.boxes:
  class_id = result.names[box.cls[0].item()]
  cords = box.xyxy[0].tolist()
  cords = [round(x) for x in cords]
  conf = round(box.conf[0].item(), 2)
  print("Object type:", class_id)
  print("Coordinates:", cords)
  print("Probability:", conf)
  print("---")

Image.fromarray(result.plot()[:,:,::-1])

We can see that the YOLO model did miss some cars that are in the 'far back' of the image.  But overall, the model did a great job of identifying multiple cars in this image.  And more importantly we can see that the identified cars are surrounded by a 'bounding box'!

Now that we are able to place bounding boxes around the car(s) recognized by the yolo model, we can re-train our YOLO model to identify a car 'crash'. 

**Please open the notebook `04-03-model-retraining.ipynb`.**